# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd
import numpy as np

We chose to concatenate the three raw datasets first and then apply the cleaning steps to the combined DataFrame. This approach ensured that all files were processed consistently and avoided duplicating the cleaning code

In [3]:
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

In [6]:
df1.shape

(4008, 11)

In [7]:
df2.shape


(996, 11)

In [8]:
df3.shape

(7070, 11)

In [34]:
# Combine the three datasets
df = pd.concat([df1,df2,df3], ignore_index=True)

In [35]:
df.shape

(12074, 13)

In [36]:
df.columns

Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Vehicle Class', 'Total Claim Amount', 'State',
       'Gender'],
      dtype='str')

We noticed that the datasets use different names for the same variables (ST/State and GENDER/Gender).

In [37]:
# Fill missing values in the State column using values from ST
df["State"] = df["ST"].fillna(df["State"])

# Fill missing values in the Gender column using values from GENDER
df["Gender"] = df["GENDER"].fillna(df["Gender"])

# Remove the old duplicate columns after merging the information
df = df.drop(columns=["ST", "GENDER"])

In [38]:
# Standardize column names

df.columns = df.columns.str.lower().str.replace(" ", "_")

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12074 entries, 0 to 12073
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9137 non-null   str    
 1   education                  9137 non-null   str    
 2   customer_lifetime_value    9130 non-null   object 
 3   income                     9137 non-null   float64
 4   monthly_premium_auto       9137 non-null   float64
 5   number_of_open_complaints  9137 non-null   object 
 6   policy_type                9137 non-null   str    
 7   vehicle_class              9137 non-null   str    
 8   total_claim_amount         9137 non-null   float64
 9   state                      9137 non-null   str    
 10  gender                     9015 non-null   str    
dtypes: float64(3), object(2), str(6)
memory usage: 1.0+ MB


In [40]:
df.head()

,customer,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount,state,gender
0,RB50392,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934,Washington,NaN
1,QZ44356,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935,Arizona,F
2,AI49188,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247,Nevada,F
3,WW63253,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344,California,M
4,GA49547,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323,Washington,M


customer_lifetime_value

In [41]:
df["customer_lifetime_value"].head()

0            NaN
1     697953.59%
2    1288743.17%
3     764586.18%
4     536307.65%
Name: customer_lifetime_value, dtype: object

In [42]:
# Remove the percentage symbol
df["customer_lifetime_value"] = (
    df["customer_lifetime_value"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

# Convert customer lifetime value to numeric
df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"],
    errors="coerce"
)

# Verify the conversion
df["customer_lifetime_value"].isna().sum()

np.int64(2944)

number_of_open_complaints

In [43]:
df["number_of_open_complaints"].head()

0    1/0/00
1    1/0/00
2    1/0/00
3    1/0/00
4    1/0/00
Name: number_of_open_complaints, dtype: object

In [44]:
# Inspect the complaint format
df["number_of_open_complaints"].unique()

array(['1/0/00', '1/2/00', '1/1/00', '1/3/00', '1/5/00', '1/4/00', nan, 0,
       2, 3, 1, 5, 4], dtype=object)

In [45]:
# Extract the middle value from the complaint format (e.g. 1/5/00 -> 5)
df["number_of_open_complaints"] = (
    df["number_of_open_complaints"]
    .astype(str)
    .str.split("/")
    .str[1]
)

# Convert number_of_open_complaints to numeric
df["number_of_open_complaints"] = pd.to_numeric(
    df["number_of_open_complaints"],
    errors="coerce"
)

# Verify the conversion
df["number_of_open_complaints"].isna().sum()

np.int64(10007)

In [46]:
df["number_of_open_complaints"].head(10)

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0
Name: number_of_open_complaints, dtype: float64

In [53]:
# Recreate the original complaint column from the raw files
raw_df = pd.concat([df1, df2, df3], ignore_index=True)

# Extract the middle value only when the value contains "/"
df["number_of_open_complaints"] = raw_df["Number of Open Complaints"].apply(
    lambda x: str(x).split("/")[1] if pd.notna(x) and "/" in str(x) else x
)

# Convert to numeric
df["number_of_open_complaints"] = pd.to_numeric(
    df["number_of_open_complaints"],
    errors="coerce"
)

# Verify
df["number_of_open_complaints"].isna().sum()

np.int64(2937)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [54]:
# Load the marketing customer analysis dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

marketing_df = pd.read_csv(url)

In [55]:
# Inspect the dimensions of the dataset
marketing_df.shape

(10910, 27)

In [56]:
# Display the first five rows
marketing_df.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [57]:
# Check data types and missing values
marketing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unnamed:_0                     10910 non-null  int64  
 1   customer                       10910 non-null  str    
 2   state                          10910 non-null  str    
 3   customer_lifetime_value        10910 non-null  float64
 4   response                       10910 non-null  str    
 5   coverage                       10910 non-null  str    
 6   education                      10910 non-null  str    
 7   effective_to_date              10910 non-null  str    
 8   employmentstatus               10910 non-null  str    
 9   gender                         10910 non-null  str    
 10  income                         10910 non-null  int64  
 11  location_code                  10910 non-null  str    
 12  marital_status                 10910 non-null  str    
 1

1. Data cleaning

In [58]:
# Check missing values
marketing_df.isna().sum()

unnamed:_0                       0
customer                         0
state                            0
customer_lifetime_value          0
response                         0
coverage                         0
education                        0
effective_to_date                0
employmentstatus                 0
gender                           0
income                           0
location_code                    0
marital_status                   0
monthly_premium_auto             0
months_since_last_claim          0
months_since_policy_inception    0
number_of_open_complaints        0
number_of_policies               0
policy_type                      0
policy                           0
renew_offer_type                 0
sales_channel                    0
total_claim_amount               0
vehicle_class                    0
vehicle_size                     0
vehicle_type                     0
month                            0
dtype: int64

In [59]:
# Check duplicate rows
marketing_df.duplicated().sum()

np.int64(0)

In [60]:
# Inspect the unnamed column
marketing_df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type', 'month'],
      dtype='str')

In [61]:
# Drop the unnecessary index column
marketing_df = marketing_df.drop(columns=["unnamed:_0"])

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [62]:
# Create a pivot table showing total revenue by sales channel
revenue_by_channel = marketing_df.pivot_table(
    values="customer_lifetime_value",
    index="sales_channel",
    aggfunc="sum"
).round(2)

revenue_by_channel

,customer_lifetime_value
sales_channel,
Agent,33057887.85
Branch,24359201.21
Call Center,17364288.37
Web,12697632.90


Analysis:
The Agent channel generated the highest total revenue, followed by Branch sales. This suggests that direct human interaction plays an important role in customer acquisition and retention. The Web channel generated the lowest revenue, indicating that online sales contribute less to the company's overall revenue compared with traditional channels.

In [63]:
# Create a pivot table showing average customer lifetime value
# by gender and education level
clv_pivot = marketing_df.pivot_table(
    values="customer_lifetime_value",
    index="education",
    columns="gender",
    aggfunc="mean"
).round(2)

clv_pivot

gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


Analysis:
Customer lifetime value varies across both gender and education levels. Customers with a High School or Below education show the highest average lifetime value, especially among females. On the other hand, customers with a Doctor degree have the lowest average CLV for both genders. Overall, the differences between males and females are relatively small within each education category, suggesting that education level may have a stronger influence on customer lifetime value than gender.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [64]:
# Create a pivot table showing the total number of complaints
# by policy type and month
complaints_pivot = marketing_df.pivot_table(
    values="number_of_open_complaints",
    index="policy_type",
    columns="month",
    aggfunc="sum"
).round(0)

complaints_pivot

month,1,2
policy_type,,
Corporate Auto,443.0,385.0
Personal Auto,1728.0,1454.0
Special Auto,87.0,95.0


In [65]:
# Convert the wide table into a long format table
complaints_long = complaints_pivot.reset_index().melt(
    id_vars="policy_type",
    var_name="month",
    value_name="total_complaints"
)

complaints_long

,policy_type,month,total_complaints
0,Corporate Auto,1,443.0
1,Personal Auto,1,1728.0
2,Special Auto,1,87.0
3,Corporate Auto,2,385.0
4,Personal Auto,2,1454.0
5,Special Auto,2,95.0


Analysis:
Personal Auto policies generated the highest number of complaints in both months, far exceeding the other policy categories. Complaints decreased from 1,728 in January to 1,454 in February. Corporate Auto policies also showed a decrease, while Special Auto complaints increased slightly. Overall, January recorded more complaints than February for most policy types.